In [58]:
import requests
from bs4 import BeautifulSoup
import json
import time
import os
from urllib.parse import urljoin

In [59]:
# Where to save the scraped articles
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# How many articles to collect per category
MAX_ARTICLES_PER_CATEGORY = 100

# Polite delay between requests in seconds
DELAY = 1.5

# Always include this so PetMD doesn't reject us as a bot
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

In [60]:
def check_robots_txt(base_url: str) -> None:
    """
    Fetch and display the site's robots.txt so we can verify scraping is permitted.

    We print the full file rather than parsing it programmatically — this way
    a human can confirm which paths are allowed or disallowed before the
    scraper runs. PetMD only blocks admin, user, and internal paths — article
    pages (/dog/*, /cat/*) are permitted.
    """
    robots_url = f"{base_url}/robots.txt"

    try:
        response = requests.get(robots_url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        print(f"robots.txt for {base_url}:\n")
        print(response.text)

    except requests.exceptions.RequestException as e:
        print(f"[WARNING] Could not retrieve robots.txt: {e}")
        print("Proceeding cautiously with polite delays.")

# Verify compliance before any scraping begins
check_robots_txt("https://www.petmd.com")

robots.txt for https://www.petmd.com:

#
# robots.txt
#
# This file is to prevent the crawling and indexing of certain parts
# of your site by web crawlers and spiders run by sites like Yahoo!
# and Google. By telling these "robots" where not to go on your site,
# you save bandwidth and server resources.
#
# This file will be ignored unless it is at the root of your host:
# Used:    http://example.com/robots.txt
# Ignored: http://example.com/site/robots.txt
#
# For more information about the robots.txt standard, see:
# http://www.robotstxt.org/robotstxt.html

User-agent: *
# CSS, JS, Images
Allow: /core/*.css$
Allow: /core/*.css?
Allow: /core/*.js$
Allow: /core/*.js?
Allow: /core/*.gif
Allow: /core/*.jpg
Allow: /core/*.jpeg
Allow: /core/*.png
Allow: /core/*.svg
Allow: /profiles/*.css$
Allow: /profiles/*.css?
Allow: /profiles/*.js$
Allow: /profiles/*.js?
Allow: /profiles/*.gif
Allow: /profiles/*.jpg
Allow: /profiles/*.jpeg
Allow: /profiles/*.png
Allow: /profiles/*.svg
# Directories
Disa

In [61]:
CATEGORIES = [
    # Dogs
    {"url": "https://www.petmd.com/dog/allergies",                    "species": "dog", "topic": "allergies"},
    {"url": "https://www.petmd.com/dog/symptoms",                     "species": "dog", "topic": "symptoms"},
    {"url": "https://www.petmd.com/dog/care",                         "species": "dog", "topic": "care"},
    {"url": "https://www.petmd.com/dog/conditions",                   "species": "dog", "topic": "conditions"},
    {"url": "https://www.petmd.com/dog/centers/nutrition",            "species": "dog", "topic": "food_safety"},
    {"url": "https://www.petmd.com/dog/emergency/poisoning-toxicity", "species": "dog", "topic": "poisoning"},
    {"url": "https://www.petmd.com/dog/behavior",                     "species": "dog", "topic": "behavior"},
    {"url": "https://www.petmd.com/dog/puppy",                        "species": "dog", "topic": "puppies"},
    {"url": "https://www.petmd.com/dog/senior",                       "species": "dog", "topic": "senior"},

    # Cats
    {"url": "https://www.petmd.com/cat/allergies",                    "species": "cat", "topic": "allergies"},
    {"url": "https://www.petmd.com/cat/symptoms",                     "species": "cat", "topic": "symptoms"},
    {"url": "https://www.petmd.com/cat/care",                         "species": "cat", "topic": "care"},
    {"url": "https://www.petmd.com/cat/conditions",                   "species": "cat", "topic": "conditions"},
    {"url": "https://www.petmd.com/cat/centers/nutrition",            "species": "cat", "topic": "food_safety"},
    {"url": "https://www.petmd.com/cat/emergency/poisoning-toxicity", "species": "cat", "topic": "poisoning"},
    {"url": "https://www.petmd.com/cat/behavior",                     "species": "cat", "topic": "behavior"},
    {"url": "https://www.petmd.com/cat/kitten",                       "species": "cat", "topic": "kittens"},
    {"url": "https://www.petmd.com/cat/senior",                       "species": "cat", "topic": "senior"},
]

In [62]:
def get_article_links(category_url: str, species: str, topic: str) -> list[str]:
    """
    Visit a PetMD category page and return up to MAX_ARTICLES_PER_CATEGORY article URLs.

    Uses static HTML parsing — works for PetMD category pages whose content
    is rendered server-side. Article URLs are identified by having at least
    3 path segments, which distinguishes them from short navigation links
    like /dog/conditions.
    """
    print(f"\nScraping category: {species} / {topic}")

    try:
        response = requests.get(category_url, headers=HEADERS, timeout=10)
        response.raise_for_status()

    except requests.exceptions.Timeout:
        print(f"  [TIMEOUT] {category_url}")
        return []

    except requests.exceptions.HTTPError as e:
        print(f"  [HTTP {e.response.status_code}] {category_url}")
        return []

    except requests.exceptions.RequestException as e:
        print(f"  [NETWORK ERROR] {category_url}: {e}")
        return []

    soup = BeautifulSoup(response.content, "html.parser")
    article_links = []
    seen = set()

    for link in soup.find_all("a", href=True):
        href = link["href"]
        # Article URLs have at least 3 path segments — filters out nav links like /dog/conditions
        if href.count("/") >= 3 and not href.startswith("http"):
            full_url = urljoin("https://www.petmd.com", href)
            if full_url not in seen:
                seen.add(full_url)
                article_links.append(full_url)
        if len(article_links) >= MAX_ARTICLES_PER_CATEGORY:
            break

    print(f"  Found {len(article_links)} article links")
    return article_links

In [63]:
def scrape_article(url: str, species: str, topic: str) -> dict | None:
    """
    Visit a single PetMD article and extract its title and body text.

    Returns a dictionary with the article's content and metadata,
    or None if the article cannot be scraped (e.g. paywalled, empty, or broken).
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()  # raises HTTPError for 4xx/5xx responses

    except requests.exceptions.Timeout:
        # The server took too long to respond — log and skip this article
        print(f"  [TIMEOUT] {url}")
        return None

    except requests.exceptions.HTTPError as e:
        # The server responded with an error code (e.g. 404 Not Found, 429 Too Many Requests)
        print(f"  [HTTP {e.response.status_code}] {url}")
        return None

    except requests.exceptions.RequestException as e:
        # Catch-all for other network problems (DNS failure, connection reset, etc.)
        print(f"  [NETWORK ERROR] {url}: {e}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    # Extract the article title from the page's main heading
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"

    # Find the article body — the class has a generated hash suffix so we match partially
    body = soup.find("div", class_=lambda c: c and "article_body" in c)

    if not body:
        print(f"  [NO BODY] {url}")
        return None

    paragraphs = body.find_all("p")
    content = " ".join(p.get_text(strip=True) for p in paragraphs)

    if not content.strip():
        print(f"  [EMPTY CONTENT] {url}")
        return None

    return {
        "title":   title,
        "content": content,
        "url":     url,
        "source":  "PetMD",
        "species": species,
        "topic":   topic,
    }

In [64]:
def run_scraper(categories):
    """Main scraper loop — runs through all categories and saves articles."""
    total_saved = 0

    for category in categories:
        url     = category["url"]
        species = category["species"]
        topic   = category["topic"]

        # get_article_links handles its own progress printing
        article_links = get_article_links(url, species, topic)
        time.sleep(DELAY)

        for article_url in article_links:
            # Use a filename based on the URL to avoid duplicates across runs
            filename = article_url.replace("https://www.petmd.com/", "").replace("/", "_") + ".json"
            filepath = os.path.join(OUTPUT_DIR, filename)

            if os.path.exists(filepath):
                print(f"  Skipping (already exists): {filename[:50]}")
                continue

            article = scrape_article(article_url, species, topic)

            if article:
                with open(filepath, "w", encoding="utf-8") as out_file:
                    json.dump(article, out_file, ensure_ascii=False, indent=2)
                total_saved += 1
                print(f"  Saved: {article['title'][:60]}")

            time.sleep(DELAY)

    print(f"\nDone. Total articles saved: {total_saved}")

# Run it
run_scraper(CATEGORIES)


Scraping category: dog / allergies
  Found 23 article links
  [NO BODY] https://www.petmd.com/dog/centers/nutrition
  [NO BODY] https://www.petmd.com/dog/emergency/poisoning-toxicity
  [NO BODY] https://www.petmd.com/dog/medications/flea-tick
  [NO BODY] https://www.petmd.com/dog/medications/heartworm
  [NO BODY] https://www.petmd.com/dog/medications/anxiety
  [NO BODY] https://www.petmd.com/news/topics/alert-recalls
  [NO BODY] https://www.petmd.com/cat/centers/nutrition
  [NO BODY] https://www.petmd.com/cat/medications/flea-tick
  [NO BODY] https://www.petmd.com/cat/medications/heartworm
  [NO BODY] https://www.petmd.com/cat/medications/anxiety
  Skipping (already exists): dog_general-health_food-allergies-vs-seasonal-alle
  Skipping (already exists): dog_conditions_skin_can-dogs-be-allergic-grass.jso
  Skipping (already exists): dog_conditions_systemic_pollen-allergies-dogs.json
  Skipping (already exists): dog_symptoms_itchy-dog.json
  Skipping (already exists): dog_general-health

In [65]:
# ── QA: Check for empty or very short articles ────────────────────────────────
# An article under 100 characters is almost certainly a scraping failure.
article_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".json")]
empty_count = 0

for filename in article_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)
    if not article["content"] or len(article["content"]) < 100:
        empty_count += 1
        print(f"  Too short: {article['title'][:60]}")

print(f"\nTotal articles: {len(article_files)}")
print(f"Empty or too short: {empty_count}")


Total articles: 339
Empty or too short: 0


In [66]:
# ── QA: Content length distribution ───────────────────────────────────────────
lengths = []
for filename in article_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)
    lengths.append(len(article["content"]))

print(f"Shortest: {min(lengths)} chars")
print(f"Longest:  {max(lengths)} chars")
print(f"Average:  {sum(lengths) // len(lengths)} chars")
print(f"Under 500 chars:  {sum(1 for l in lengths if l < 500)}")
print(f"Under 1000 chars: {sum(1 for l in lengths if l < 1000)}")

Shortest: 636 chars
Longest:  36592 chars
Average:  5867 chars
Under 500 chars:  0
Under 1000 chars: 3


In [67]:
# ── QA: List all articles under 1000 characters ───────────────────────────────
for filename in article_files:
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)
    if len(article["content"]) < 1000:
        print(f"{article['title']} — {len(article['content'])} chars — {article['species']}/{article['topic']}")

Adhesions of the Eye’s Iris and Swelling of Eye in Cats — 809 chars — cat/conditions
Ameba Infection in Cats — 945 chars — cat/conditions
9 Best Probiotics for Dogs in 2025, Recommended By Vets — 636 chars — dog/senior


In [68]:
# ── QA: Article distribution by species and topic ─────────────────────────────
from collections import Counter

distribution   = Counter()
missing_fields = []

for filename in os.listdir(OUTPUT_DIR):
    if not filename.endswith(".json"):
        continue
    with open(os.path.join(OUTPUT_DIR, filename), encoding="utf-8") as fp:
        article = json.load(fp)

    distribution[(article.get("species", "MISSING"), article.get("topic", "MISSING"))] += 1

    for field in ["title", "content", "url", "source", "species", "topic"]:
        if not article.get(field):
            missing_fields.append((filename, field))

print(f"{'SPECIES':<14} {'TOPIC':<18} {'COUNT':>6}")
print("-" * 40)
for (species, topic), count in sorted(distribution.items()):
    print(f"{species:<14} {topic:<18} {count:>6}")
print("-" * 40)
print(f"{'TOTAL':<33} {sum(distribution.values()):>6}")

if missing_fields:
    print(f"\nWARNING: {len(missing_fields)} missing field(s) detected")
    for fname, field in missing_fields[:10]:
        print(f"  {fname} — missing '{field}'")
else:
    print("\nAll articles have required fields. ✓")

SPECIES        TOPIC               COUNT
----------------------------------------
cat            allergies               7
cat            behavior               14
cat            care                    5
cat            conditions             89
cat            food_safety            10
cat            general                 1
cat            kittens                10
cat            senior                 14
cat            symptoms               11
dog            allergies               9
dog            behavior               11
dog            care                    9
dog            conditions             89
dog            food_safety             9
dog            poisoning              13
dog            puppies                14
dog            senior                 12
dog            symptoms               12
----------------------------------------
TOTAL                                339

All articles have required fields. ✓
